In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import holidays
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
df_pre = pd.read_csv("../../Data/은평구_스테이션_군집화_1차_자전거_이용_정보.csv")
df_pre

In [ ]:
# %pip install holidays

In [ ]:
df = pd.read_csv("../../Data/2024_st_hhj.csv")

In [ ]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
df

In [ ]:
df_2025 = pd.read_parquet("../../Data/sort_data/2025_data.parquet")
df_2024 = pd.read_parquet("../../Data/sort_data/2024_data.parquet")

df = pd.concat([df_2024, df_2025], axis=0)
kr = holidays.KR()

In [ ]:
df

In [ ]:
# 1. 시작 대여소 기준으로만 필터링 (순수 대여량 예측 목적)
target_station = ['ST-1483', 'ST-1481', 'ST-453']
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['year'] = df['기준_날짜'].dt.year
df['weekday'] = df['기준_날짜'].dt.dayofweek
df['day_type'] = np.where(df['weekday'] < 5, 0, 1)  # 평일: 0, 주말: 1
df = df[
    (df['전체_이용_분'] >= pd.Timedelta(minutes=5)) &
    (df['전체_이용_거리'] >= 5)
]

df['datetime'] = pd.to_datetime(df['기준_날짜']) + pd.to_timedelta(df['시간대'], unit='h')
df = df[df['시작_대여소_ID'].isin(target_station)].copy()
df_hourly = df.groupby(['시작_대여소_ID', 'datetime','year']).agg({
    '전체_건수': 'sum',
    '온도': 'mean',
    '습도': 'mean',
    '불쾌지수': 'mean',
    '강수량': 'mean',
    '적설량': 'mean'
}).reset_index()

In [ ]:
df

In [ ]:
df.drop(columns = ['weekday', '전체_이용_분', '전체_이용_거리', '불쾌지수'],  inplace=True)

In [ ]:
df.head()

In [ ]:
# 1. 시간대와 요일 정보 다시 추출
df_hourly['hour'] = df_hourly['datetime'].dt.hour
df_hourly['weekday'] = df_hourly['datetime'].dt.dayofweek
df_hourly['is_weekend'] = np.where(df_hourly['weekday'] < 5, 0, 1)
df_hourly['is_holiday'] = df_hourly['datetime'].dt.date.isin(kr)

# 2. 대여소 ID를 숫자로 변환 (HGB가 대여소별 특징을 알게 함)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_hourly['station_idx'] = le.fit_transform(df_hourly['시작_대여소_ID'])
df_hourly = df_hourly.sort_values(['시작_대여소_ID', 'datetime'])

df_hourly['lag_1h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(1)
df_hourly['lag_2h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(2)

df_hourly['lag_24h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(24)

df_hourly['rolling_3h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].transform(lambda x: x.shift(1).rolling(3).mean())

df_hourly = df_hourly.dropna()

df_hourly['hour_sin'] = np.sin(2 * np.pi * df_hourly['hour'] / 24)
df_hourly['hour_cos'] = np.cos(2 * np.pi * df_hourly['hour'] / 24)


In [ ]:
all_stations = df_hourly['시작_대여소_ID'].unique()
all_times = pd.date_range(start=df_hourly['datetime'].min(), 
                          end=df_hourly['datetime'].max(), 
                          freq='h')

multi_idx = pd.MultiIndex.from_product([all_stations, all_times], 
                                       names=['시작_대여소_ID', 'datetime'])
df_full = pd.DataFrame(index=multi_idx).reset_index()

df_hourly = pd.merge(df_full, df_hourly, on=['시작_대여소_ID', 'datetime'], how='left')

df_hourly['전체_건수'] = df_hourly['전체_건수'].fillna(0) # 대여 없는 시간은 0건
df_hourly['year'] = df_hourly['datetime'].dt.year
df_hourly[['온도', '습도', '불쾌지수', '강수량', '적설량']] = \
    df_hourly.groupby('시작_대여소_ID')[['온도', '습도', '불쾌지수', '강수량', '적설량']].ffill()


df_hourly = pd.get_dummies(df_hourly, columns=['시작_대여소_ID'], prefix='st')

# 2. 새로운 피처 리스트 생성 (기 station_idx 제외하고 새로 생긴 st_... 컬럼들 추가)
station_cols = [col for col in df_hourly.columns if col.startswith('st_')]
target = '전체_건수'
features = [
    # 'hour',
      'weekday', 
    # 'is_weekend', 'is_holiday',
    '온도', '습도',
      # '불쾌지수',
        '강수량', '적설량',
    'lag_1h', 
    # 'lag_2h',
    #   'rolling_3h',
      'lag_24h',
    'hour_sin',
    #   'hour_cos'
] 


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import HistGradientBoostingRegressor



train = df_hourly[df_hourly['year'] == 2024]
test  = df_hourly[df_hourly['year'] == 2025]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

x_input, x_valid, y_input, y_valid = train_test_split(
    X_train, y_train, random_state=42, test_size=0.2
)

hgb = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=600,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)
from sklearn.model_selection import TimeSeriesSplit, cross_validate

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_validate(
    hgb,
    X_train,
    np.log1p(y_train),
    cv=tscv,
    n_jobs=1,
    return_train_score=True
)
# scores = cross_validate(
#     hgb,
#     X_train,
#     np.log1p(y_train),
#     n_jobs=1,
#     return_train_score=True
# )

# log 변환 학습
hgb.fit(x_input, np.log1p(y_input))

# score는 log scale끼리 비교해야 의미 있음
print("train score(log):", hgb.score(x_input, np.log1p(y_input)))
print("valid score(log):", hgb.score(x_valid, np.log1p(y_valid)))

print("All Train score(log):", hgb.score(X_train, np.log1p(y_train)))
print("Test score(log):", hgb.score(X_test, np.log1p(y_test)))
# 원래 값으로 복원해서 평가
pred_input = np.expm1(hgb.predict(x_input))
pred_valid = np.expm1(hgb.predict(x_valid))
pred_train = np.expm1(hgb.predict(X_train))
pred_test = np.expm1(hgb.predict(X_test))

print('========================')

print("Input MAE:", mean_absolute_error(y_input, pred_input))
print("Input RMSE:", np.sqrt(mean_squared_error(y_input, pred_input)))
print("Input R2:", r2_score(y_input, pred_input))

print('========================')

print("VALID MAE:", mean_absolute_error(y_valid, pred_valid))
print("VALID RMSE:", np.sqrt(mean_squared_error(y_valid, pred_valid)))
print("VALID R2:", r2_score(y_valid, pred_valid))

print('========================')
print("Train MAE:", mean_absolute_error(y_train, pred_train))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, pred_train)))
print("Train R2:", r2_score(y_train, pred_train))

print('========================')
print("TEST MAE:", mean_absolute_error(y_test, pred_test))
print("TEST RMSE:", np.sqrt(mean_squared_error(y_test, pred_test)))
print("TEST R2:", r2_score(y_test, pred_test))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 분석에 사용할 변수들만 추출
corr_df = df_hourly[features + [target]]

plt.figure(figsize=(12, 8))
sns.heatmap(corr_df.corr(), annot=True, cmap='RdYlGn', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# 테스트 데이터셋을 기준으로 측정합니다.
result = permutation_importance(hgb, X_test, y_test, n_repeats=10, random_state=42)

# 시각화를 위해 정렬
sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.barh(np.array(features)[sorted_idx], result.importances_mean[sorted_idx])
plt.xlabel("Permutation Importance (Decrease in R2 Score)")
plt.title("Feature Importance - HGB Model")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
result = permutation_importance(
    hgb, X_test, np.log1p(y_test), n_repeats=10, random_state=42
)

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import holidays
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# -------------------------------------------------
# 0. 데이터 로드
# -------------------------------------------------
df_2024 = pd.read_parquet("../../Data/sort_data/2024_data.parquet")
df_2025 = pd.read_parquet("../../Data/sort_data/2025_data.parquet")
df = pd.concat([df_2024, df_2025], axis=0).copy()

kr = holidays.KR()

# -------------------------------------------------
# 1. 기본 전처리
# -------------------------------------------------
target_station = ['ST-1483', 'ST-1481', 'ST-453']

df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['year'] = df['기준_날짜'].dt.year
df['weekday'] = df['기준_날짜'].dt.dayofweek
df['day_type'] = np.where(df['weekday'] < 5, 0, 1)   # 평일 0 / 주말 1

# 전체_이용_분이 숫자 분이라고 가정
# df = df[
#     (df['전체_이용_분'] >= 5) &
#     (df['전체_이용_거리'] > 0)
# ].copy()
if pd.api.types.is_timedelta64_dtype(df['전체_이용_분']):
    duration_mask = df['전체_이용_분'] >= pd.Timedelta(minutes=5)
else:
    duration_mask = df['전체_이용_분'] >= 5

df = df[
    duration_mask &
    (df['전체_이용_거리'] > 0)
].copy()


df['datetime'] = df['기준_날짜'] + pd.to_timedelta(df['시간대'], unit='h')

# 시작 대여소 필터
df = df[df['시작_대여소_ID'].isin(target_station)].copy()

# -------------------------------------------------
# 2. 시간 단위 집계
# -------------------------------------------------
df_hourly = (
    df.groupby(['시작_대여소_ID', 'datetime'], as_index=False)
      .agg({
          '전체_건수': 'sum',
          '온도': 'mean',
          '습도': 'mean',
          '불쾌지수': 'mean',
          '강수량': 'mean',
          '적설량': 'mean'
      })
)

# -------------------------------------------------
# 3. full timeline 먼저 생성
#    -> 이 다음에 lag/rolling 생성해야 진짜 1시간 전 값이 됨
# -------------------------------------------------
all_stations = df_hourly['시작_대여소_ID'].unique()
all_times = pd.date_range(
    start=df_hourly['datetime'].min(),
    end=df_hourly['datetime'].max(),
    freq='h'
)

multi_idx = pd.MultiIndex.from_product(
    [all_stations, all_times],
    names=['시작_대여소_ID', 'datetime']
)
df_full = pd.DataFrame(index=multi_idx).reset_index()

df_hourly = pd.merge(
    df_full,
    df_hourly,
    on=['시작_대여소_ID', 'datetime'],
    how='left'
)

# 대여 없는 시간은 0건
df_hourly['전체_건수'] = df_hourly['전체_건수'].fillna(0)

# 날씨는 같은 station 내에서 시간순 forward fill
weather_cols = ['온도', '습도', '불쾌지수', '강수량', '적설량']
df_hourly = df_hourly.sort_values(['시작_대여소_ID', 'datetime']).copy()
df_hourly[weather_cols] = (
    df_hourly.groupby('시작_대여소_ID')[weather_cols]
    .ffill()
    .bfill()
)

# -------------------------------------------------
# 4. 시간 파생변수
# -------------------------------------------------
df_hourly['year'] = df_hourly['datetime'].dt.year
df_hourly['month'] = df_hourly['datetime'].dt.month
df_hourly['day'] = df_hourly['datetime'].dt.day
df_hourly['hour'] = df_hourly['datetime'].dt.hour
df_hourly['weekday'] = df_hourly['datetime'].dt.dayofweek

df_hourly['is_weekend'] = (df_hourly['weekday'] >= 5).astype(int)
df_hourly['is_holiday'] = df_hourly['datetime'].dt.date.isin(kr).astype(int)

df_hourly['hour_sin'] = np.sin(2 * np.pi * df_hourly['hour'] / 24)
df_hourly['hour_cos'] = np.cos(2 * np.pi * df_hourly['hour'] / 24)

# -------------------------------------------------
# 5. lag / rolling
#    반드시 미래 안 보게 shift 먼저
# -------------------------------------------------
df_hourly = df_hourly.sort_values(['시작_대여소_ID', 'datetime']).copy()

grp = df_hourly.groupby('시작_대여소_ID')['전체_건수']

df_hourly['lag_1h'] = grp.shift(1)
df_hourly['lag_2h'] = grp.shift(2)
df_hourly['lag_24h'] = grp.shift(24)

df_hourly['rolling_3h'] = grp.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df_hourly['rolling_6h'] = grp.shift(1).rolling(6).mean().reset_index(level=0, drop=True)
df_hourly['rolling_24h'] = grp.shift(1).rolling(24).mean().reset_index(level=0, drop=True)

# 초기 구간은 과거값이 없으니 NaN 유지 후 제거
df_hourly = df_hourly.dropna().copy()

# -------------------------------------------------
# 6. station one-hot 추가
# -------------------------------------------------
df_hourly = pd.get_dummies(df_hourly, columns=['시작_대여소_ID'], prefix='st')
station_cols = [col for col in df_hourly.columns if col.startswith('st_')]

# -------------------------------------------------
# 7. feature 정의
#    month/day/hour은 일단 제외하고 네 기존 흐름 유지
# -------------------------------------------------
target = '전체_건수'

features = [
    'weekday',
    'is_weekend',
    'is_holiday',
    '온도',
    '습도',
    '강수량',
    '적설량',
    'lag_1h',
    'lag_2h',
    'lag_24h',
    'rolling_3h',
    'rolling_6h',
    'rolling_24h',
    'hour_sin',
    'hour_cos'
] + station_cols

# -------------------------------------------------
# 8. 시간순 분리
#    랜덤 분리 제거
# -------------------------------------------------
train = df_hourly[df_hourly['datetime'] < '2024-10-01'].copy()
valid = df_hourly[(df_hourly['datetime'] >= '2024-10-01') & (df_hourly['datetime'] < '2025-01-01')].copy()
test  = df_hourly[df_hourly['year'] == 2025].copy()

X_train = train[features]
y_train = train[target]

X_valid = valid[features]
y_valid = valid[target]

X_test = test[features]
y_test = test[target]

print("train:", X_train.shape, y_train.shape)
print("valid:", X_valid.shape, y_valid.shape)
print("test :", X_test.shape, y_test.shape)

# -------------------------------------------------
# 9. 모델
# -------------------------------------------------
hgb = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=600,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)

# -------------------------------------------------
# 10. TimeSeriesSplit CV
# -------------------------------------------------
tscv = TimeSeriesSplit(n_splits=5)

scores = cross_validate(
    hgb,
    X_train,
    np.log1p(y_train),
    cv=tscv,
    scoring='r2',
    n_jobs=1,
    return_train_score=True
)

print("\n================ CV (log target) ================")
print("Train CV mean:", scores['train_score'].mean())
print("Valid CV mean:", scores['test_score'].mean())

# -------------------------------------------------
# 11. 학습
# -------------------------------------------------
hgb.fit(X_train, np.log1p(y_train))

print("\n================ SCORE (log scale) ================")
print("Train score(log):", hgb.score(X_train, np.log1p(y_train)))
print("Valid score(log):", hgb.score(X_valid, np.log1p(y_valid)))
print("Test score(log):", hgb.score(X_test, np.log1p(y_test)))

# -------------------------------------------------
# 12. 원래 값으로 복원 후 평가
# -------------------------------------------------
pred_train = np.expm1(hgb.predict(X_train))
pred_valid = np.expm1(hgb.predict(X_valid))
pred_test  = np.expm1(hgb.predict(X_test))

def print_metrics(name, y_true, y_pred):
    print(f"\n================ {name} ================")
    print("MAE :", mean_absolute_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print("R2  :", r2_score(y_true, y_pred))

print_metrics("TRAIN", y_train, pred_train)
print_metrics("VALID", y_valid, pred_valid)
print_metrics("TEST", y_test, pred_test)

# -------------------------------------------------
# 13. 상관행렬
# -------------------------------------------------
corr_df = df_hourly[features + [target]]

plt.figure(figsize=(14, 10))
sns.heatmap(corr_df.corr(), annot=True, cmap='RdYlGn', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

# -------------------------------------------------
# 14. permutation importance
#     log scale 기준으로 맞춰서 계산
# -------------------------------------------------
result = permutation_importance(
    hgb,
    X_test,
    np.log1p(y_test),   # 중요: log scale로 맞춤
    n_repeats=10,
    random_state=42,
    scoring='r2'
)

sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.barh(np.array(features)[sorted_idx], result.importances_mean[sorted_idx])
plt.xlabel("Permutation Importance (Decrease in R2, log scale)")
plt.title("Feature Importance - HGB Model")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# -------------------------------------------------
# 15. 실제값 vs 예측값 / residual
# -------------------------------------------------
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred_test, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Test Actual vs Predicted")
plt.grid(True)
plt.show()

residuals = y_test - pred_test

plt.figure(figsize=(8, 5))
plt.scatter(pred_test, residuals, alpha=0.3)
plt.axhline(0, linestyle='--')
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Test Residual Plot")
plt.grid(True)
plt.show()

# -------------------------------------------------
# 16. 샘플 구간 시각화
# -------------------------------------------------
sample_n = 300
plt.figure(figsize=(14, 5))
plt.plot(y_test.iloc[:sample_n].values, label='Actual')
plt.plot(pred_test[:sample_n], label='Predicted')
plt.title(f"Test Sample Plot (first {sample_n})")
plt.xlabel("Index")
plt.ylabel("Count")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 날짜형 확인
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 연/월 추출
df['year'] = df['기준_날짜'].dt.year
df['month'] = df['기준_날짜'].dt.month

# 연도-월별 전체 건수 합계
monthly_sum = (
    df.groupby(['year', 'month'], as_index=False)['전체_건수']
      .sum()
      .sort_values(['year', 'month'])
)

print(monthly_sum)

# 선그래프
plt.figure(figsize=(12, 6))

for year in monthly_sum['year'].unique():
    temp = monthly_sum[monthly_sum['year'] == year]
    plt.plot(temp['month'], temp['전체_건수'], marker='o', label=f'{year}')

plt.title('연도별 월별 전체 건수')
plt.xlabel('월')
plt.ylabel('전체_건수 합계')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
monthly_norm = monthly_sum.copy()

monthly_norm['정규화_건수'] = (
    monthly_norm.groupby('year')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

plt.figure(figsize=(12, 6))

for year in monthly_norm['year'].unique():
    temp = monthly_norm[monthly_norm['year'] == year]
    plt.plot(temp['month'], temp['정규화_건수'], marker='o', label=f'{year}')

plt.title('연도별 월별 패턴 비교(정규화)')
plt.xlabel('월')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
monthly_mean = (
    df.groupby(['year', 'month'], as_index=False)['전체_건수']
      .mean()
      .sort_values(['year', 'month'])
)

plt.figure(figsize=(12, 6))

for year in monthly_mean['year'].unique():
    temp = monthly_mean[monthly_mean['year'] == year]
    plt.plot(temp['month'], temp['전체_건수'], marker='o', label=f'{year}')

plt.title('연도별 월별 평균 전체 건수')
plt.xlabel('월')
plt.ylabel('전체_건수 평균')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 날짜형 변환
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 연도 하나만 보고 싶으면 예: 2024
df_year = df[df['기준_날짜'].dt.year == 2024].copy()

# 월, 일 추출
df_year['month'] = df_year['기준_날짜'].dt.month
df_year['day'] = df_year['기준_날짜'].dt.day

# 월-일별 전체 건수 합계
month_day_sum = (
    df_year.groupby(['month', 'day'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'day'])
)

print(month_day_sum.head())

# 그래프
plt.figure(figsize=(14, 7))

for month in sorted(month_day_sum['month'].unique()):
    temp = month_day_sum[month_day_sum['month'] == month]
    plt.plot(temp['day'], temp['전체_건수'], marker='o', label=f'{month}월')

plt.title('2024년 월별 일자 패턴')
plt.xlabel('일자')
plt.ylabel('전체_건수 합계')
plt.xticks(range(1, 32))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df_year = df[df['기준_날짜'].dt.year == 2024].copy()

df_year['month'] = df_year['기준_날짜'].dt.month
df_year['day'] = df_year['기준_날짜'].dt.day

month_day_sum = (
    df_year.groupby(['month', 'day'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'day'])
)

month_day_sum['정규화_건수'] = (
    month_day_sum.groupby('month')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

plt.figure(figsize=(14, 7))
for month in sorted(month_day_sum['month'].unique()):
    temp = month_day_sum[month_day_sum['month'] == month]
    plt.plot(temp['day'], temp['정규화_건수'], marker='o', label=f'{month}월')

plt.title('월별 일자 패턴 비교')
plt.xlabel('일자')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(1, 32))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 날짜형 변환
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 2024년 + 4월~9월만 필터링
df_2024 = df[
    (df['기준_날짜'].dt.year == 2024) &
    (df['기준_날짜'].dt.month >= 4) &
    (df['기준_날짜'].dt.month <= 9)
].copy()

# 월, 일 추출
df_2024['month'] = df_2024['기준_날짜'].dt.month
df_2024['day'] = df_2024['기준_날짜'].dt.day

# 월-일별 전체 건수 합계
month_day_sum = (
    df_2024.groupby(['month', 'day'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'day'])
)

# 월별 정규화
month_day_sum['정규화_건수'] = (
    month_day_sum.groupby('month')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

# 그래프
plt.figure(figsize=(14, 7))

for month in sorted(month_day_sum['month'].unique()):
    temp = month_day_sum[month_day_sum['month'] == month]
    plt.plot(temp['day'], temp['정규화_건수'], marker='o', label=f'{month}월')

plt.title('2024년 4월~9월 월별 일자 패턴 비교')
plt.xlabel('일자')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(1, 32))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 날짜형 변환
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 2024년 전체만 사용
df_2024 = df[df['기준_날짜'].dt.year == 2024].copy()

# 월 / 일 / 요일
df_2024['month'] = df_2024['기준_날짜'].dt.month
df_2024['day'] = df_2024['기준_날짜'].dt.day
df_2024['weekday'] = df_2024['기준_날짜'].dt.dayofweek   # 월=0, 일=6

# 날짜 단위 달력 테이블
calendar_df = (
    df_2024[['기준_날짜', 'month', 'weekday']]
    .drop_duplicates()
    .sort_values('기준_날짜')
    .copy()
)

# 해당 월 안에서 "몇 번째 월요일/화요일..."인지
calendar_df['weekday_nth'] = calendar_df.groupby(['month', 'weekday']).cumcount() + 1

# 원본 데이터에 붙이기
df_2024 = df_2024.merge(
    calendar_df[['기준_날짜', 'weekday_nth']],
    on='기준_날짜',
    how='left'
)

# x축 라벨
weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df_2024['x_label'] = (
    df_2024['weekday_nth'].astype(str) + '주차_' + df_2024['weekday'].map(weekday_map)
)

# 월별 x_label 단위 전체 건수 합계
plot_df = (
    df_2024.groupby(['month', 'weekday_nth', 'weekday', 'x_label'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'weekday_nth', 'weekday'])
)

# 월별 정규화
plot_df['정규화_건수'] = (
    plot_df.groupby('month')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

# x축 순서 고정
x_order = (
    plot_df[['weekday_nth', 'weekday', 'x_label']]
    .drop_duplicates()
    .sort_values(['weekday_nth', 'weekday'])['x_label']
    .tolist()
)

# x축 위치 숫자로 변환
x_pos_map = {label: i for i, label in enumerate(x_order)}

# 월별 색상 고정 (12개 모두 다르게)
cmap = plt.get_cmap('tab20')
month_colors = {month: cmap(i) for i, month in enumerate(sorted(plot_df['month'].unique()))}

# 그래프
plt.figure(figsize=(20, 9))

for month in sorted(plot_df['month'].unique()):
    temp = plot_df[plot_df['month'] == month].copy()
    temp['x_label'] = pd.Categorical(temp['x_label'], categories=x_order, ordered=True)
    temp = temp.sort_values('x_label')
    temp['x_num'] = temp['x_label'].map(x_pos_map)

    plt.plot(
        temp['x_num'],
        temp['정규화_건수'],
        marker='o',
        label=f'{month}월',
        color=month_colors[month],
        linewidth=2
    )

    # 선 끝점에 월 표시
    last_x = temp['x_num'].iloc[-1]
    last_y = temp['정규화_건수'].iloc[-1]
    plt.text(
        last_x + 0.2,
        last_y,
        f'{month}월',
        color=month_colors[month],
        fontsize=9,
        va='center'
    )

plt.title('2024년 전체 월 요일 정렬 패턴 비교')
plt.xlabel('주차-요일')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(len(x_order)), x_order, rotation=45)
plt.grid(True, alpha=0.3)

# 범례는 겹치면 너무 커질 수 있어서 바깥으로 뺌
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 보고 싶은 달만 지정
# 예: 4,5,6,9월만 보기
# -----------------------------
selected_months = [1,2,12,11]

# 날짜형 변환
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 2024년 + 선택한 달만 사용
df_2024 = df[
    (df['기준_날짜'].dt.year == 2024) &
    (df['기준_날짜'].dt.month.isin(selected_months))
].copy()

# 월 / 일 / 요일
df_2024['month'] = df_2024['기준_날짜'].dt.month
df_2024['weekday'] = df_2024['기준_날짜'].dt.dayofweek   # 월=0, 일=6

# 날짜 단위 달력 테이블
calendar_df = (
    df_2024[['기준_날짜', 'month', 'weekday']]
    .drop_duplicates()
    .sort_values('기준_날짜')
    .copy()
)

# 해당 월 안에서 몇 번째 같은 요일인지
calendar_df['weekday_nth'] = calendar_df.groupby(['month', 'weekday']).cumcount() + 1

# 원본 데이터에 붙이기
df_2024 = df_2024.merge(
    calendar_df[['기준_날짜', 'weekday_nth']],
    on='기준_날짜',
    how='left'
)

# x축 라벨
weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df_2024['x_label'] = (
    df_2024['weekday_nth'].astype(str) + '주차_' + df_2024['weekday'].map(weekday_map)
)

# 월별 x_label 단위 전체 건수 합계
plot_df = (
    df_2024.groupby(['month', 'weekday_nth', 'weekday', 'x_label'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'weekday_nth', 'weekday'])
)

# 월별 정규화
plot_df['정규화_건수'] = (
    plot_df.groupby('month')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

# x축 순서 고정
x_order = (
    plot_df[['weekday_nth', 'weekday', 'x_label']]
    .drop_duplicates()
    .sort_values(['weekday_nth', 'weekday'])['x_label']
    .tolist()
)

x_pos_map = {label: i for i, label in enumerate(x_order)}

# 색상
cmap = plt.get_cmap('tab20')
month_colors = {month: cmap(i) for i, month in enumerate(sorted(plot_df['month'].unique()))}

# 그래프
plt.figure(figsize=(18, 8))

for month in sorted(plot_df['month'].unique()):
    temp = plot_df[plot_df['month'] == month].copy()
    temp['x_label'] = pd.Categorical(temp['x_label'], categories=x_order, ordered=True)
    temp = temp.sort_values('x_label')
    temp['x_num'] = temp['x_label'].map(x_pos_map)

    plt.plot(
        temp['x_num'],
        temp['정규화_건수'],
        marker='o',
        label=f'{month}월',
        color=month_colors[month],
        linewidth=2
    )

    # 선 끝점에 월 표시
    plt.text(
        temp['x_num'].iloc[-1] + 0.2,
        temp['정규화_건수'].iloc[-1],
        f'{month}월',
        color=month_colors[month],
        fontsize=10,
        va='center'
    )

plt.title(f'2024년 선택 월 요일 정렬 패턴 비교: {selected_months}')
plt.xlabel('주차-요일')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(len(x_order)), x_order, rotation=45)
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 보고 싶은 달만 지정
# 예: 4,5,6,9월만 보기
# -----------------------------
selected_months = [7,8]

# 날짜형 변환
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 2024년 + 선택한 달만 사용
df_2024 = df[
    (df['기준_날짜'].dt.year == 2024) &
    (df['기준_날짜'].dt.month.isin(selected_months))
].copy()

# 월 / 요일
df_2024['month'] = df_2024['기준_날짜'].dt.month
df_2024['weekday'] = df_2024['기준_날짜'].dt.dayofweek   # 월=0, 일=6

# 날짜 단위 달력 테이블
calendar_df = (
    df_2024[['기준_날짜', 'month', 'weekday']]
    .drop_duplicates()
    .sort_values('기준_날짜')
    .copy()
)

# 해당 월 안에서 몇 번째 같은 요일인지
calendar_df['weekday_nth'] = calendar_df.groupby(['month', 'weekday']).cumcount() + 1

# 원본 데이터에 붙이기
df_2024 = df_2024.merge(
    calendar_df[['기준_날짜', 'weekday_nth']],
    on='기준_날짜',
    how='left'
)

# x축 라벨
weekday_map = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
df_2024['x_label'] = (
    df_2024['weekday_nth'].astype(str) + '주차_' + df_2024['weekday'].map(weekday_map)
)

# 월별 x_label 단위 전체 건수 합계
plot_df = (
    df_2024.groupby(['month', 'weekday_nth', 'weekday', 'x_label'], as_index=False)['전체_건수']
    .sum()
    .sort_values(['month', 'weekday_nth', 'weekday'])
)

# x축 순서 고정
x_order = (
    plot_df[['weekday_nth', 'weekday', 'x_label']]
    .drop_duplicates()
    .sort_values(['weekday_nth', 'weekday'])['x_label']
    .tolist()
)

x_pos_map = {label: i for i, label in enumerate(x_order)}

# 색상
cmap = plt.get_cmap('tab20')
month_colors = {month: cmap(i) for i, month in enumerate(sorted(plot_df['month'].unique()))}

# 그래프
plt.figure(figsize=(18, 8))

for month in sorted(plot_df['month'].unique()):
    temp = plot_df[plot_df['month'] == month].copy()
    temp['x_label'] = pd.Categorical(temp['x_label'], categories=x_order, ordered=True)
    temp = temp.sort_values('x_label')
    temp['x_num'] = temp['x_label'].map(x_pos_map)

    plt.plot(
        temp['x_num'],
        temp['전체_건수'],
        marker='o',
        label=f'{month}월',
        color=month_colors[month],
        linewidth=2
    )

    # 선 끝점에 월 표시
    plt.text(
        temp['x_num'].iloc[-1] + 0.2,
        temp['전체_건수'].iloc[-1],
        f'{month}월',
        color=month_colors[month],
        fontsize=10,
        va='center'
    )

plt.title(f'2024년 선택 월 요일 정렬 패턴 비교 (Raw): {selected_months}')
plt.xlabel('주차-요일')
plt.ylabel('전체_건수')
plt.xticks(range(len(x_order)), x_order, rotation=45)
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

df_2024 = df[df['기준_날짜'].dt.year == 2024].copy()
df_2024['month'] = df_2024['기준_날짜'].dt.month
df_2024['day'] = df_2024['기준_날짜'].dt.day
df_2024['weekday'] = df_2024['기준_날짜'].dt.dayofweek   # 월=0, 일=6
df_2024['is_weekend'] = (df_2024['weekday'] >= 5).astype(int)

# 혹시 hour 컬럼이 없고 시간대 컬럼이 있으면 맞춰주기
if 'hour' not in df_2024.columns and '시간대' in df_2024.columns:
    df_2024['hour'] = df_2024['시간대']

In [ ]:
month_hour_avg = (
    df_2024.groupby(['month', 'hour'], as_index=False)['전체_건수']
    .mean()
    .sort_values(['month', 'hour'])
)

plt.figure(figsize=(14, 7))

for month in sorted(month_hour_avg['month'].unique()):
    temp = month_hour_avg[month_hour_avg['month'] == month]
    plt.plot(temp['hour'], temp['전체_건수'], marker='o', label=f'{month}월')

plt.title('2024년 월별 시간대 평균 패턴 (Raw)')
plt.xlabel('시간대')
plt.ylabel('전체_건수 평균')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
month_hour_avg_norm = month_hour_avg.copy()

month_hour_avg_norm['정규화_건수'] = (
    month_hour_avg_norm.groupby('month')['전체_건수']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0)
)

plt.figure(figsize=(14, 7))

for month in sorted(month_hour_avg_norm['month'].unique()):
    temp = month_hour_avg_norm[month_hour_avg_norm['month'] == month]
    plt.plot(temp['hour'], temp['정규화_건수'], marker='o', label=f'{month}월')

plt.title('2024년 월별 시간대 평균 패턴 (정규화)')
plt.xlabel('시간대')
plt.ylabel('정규화된 전체_건수')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
weekday_hour_avg = (
    df_2024[df_2024['is_weekend'] == 0]
    .groupby(['month', 'hour'], as_index=False)['전체_건수']
    .mean()
    .sort_values(['month', 'hour'])
)

plt.figure(figsize=(14, 7))

for month in sorted(weekday_hour_avg['month'].unique()):
    temp = weekday_hour_avg[weekday_hour_avg['month'] == month]
    plt.plot(temp['hour'], temp['전체_건수'], marker='o', label=f'{month}월')

plt.title('2024년 월별 평일 시간대 평균 패턴')
plt.xlabel('시간대')
plt.ylabel('전체_건수 평균')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
weekend_hour_avg = (
    df_2024[df_2024['is_weekend'] == 1]
    .groupby(['month', 'hour'], as_index=False)['전체_건수']
    .mean()
    .sort_values(['month', 'hour'])
)

plt.figure(figsize=(14, 7))

for month in sorted(weekend_hour_avg['month'].unique()):
    temp = weekend_hour_avg[weekend_hour_avg['month'] == month]
    plt.plot(temp['hour'], temp['전체_건수'], marker='o', label=f'{month}월')

plt.title('2024년 월별 주말 시간대 평균 패턴')
plt.xlabel('시간대')
plt.ylabel('전체_건수 평균')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend(ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
selected_month = 4   # 보고 싶은 달 바꿔가면서 보기

temp_df = (
    df_2024[df_2024['month'] == selected_month]
    .groupby(['is_weekend', 'hour'], as_index=False)['전체_건수']
    .mean()
    .sort_values(['is_weekend', 'hour'])
)

plt.figure(figsize=(12, 6))

for flag, label in [(0, '평일'), (1, '주말')]:
    temp = temp_df[temp_df['is_weekend'] == flag]
    plt.plot(temp['hour'], temp['전체_건수'], marker='o', label=label)

plt.title(f'2024년 {selected_month}월 평일/주말 시간대 평균 패턴')
plt.xlabel('시간대')
plt.ylabel('전체_건수 평균')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_2024['commute_morning'] = df_2024['hour'].isin([7, 8, 9]).astype(int)
df_2024['commute_evening'] = df_2024['hour'].isin([17, 18, 19, 20]).astype(int)

monthly_total = (
    df_2024.groupby('month', as_index=False)['전체_건수']
    .sum()
    .rename(columns={'전체_건수': 'month_total'})
)

monthly_morning = (
    df_2024[df_2024['commute_morning'] == 1]
    .groupby('month', as_index=False)['전체_건수']
    .sum()
    .rename(columns={'전체_건수': 'morning_total'})
)

monthly_evening = (
    df_2024[df_2024['commute_evening'] == 1]
    .groupby('month', as_index=False)['전체_건수']
    .sum()
    .rename(columns={'전체_건수': 'evening_total'})
)

commute_ratio_df = monthly_total.merge(monthly_morning, on='month', how='left')
commute_ratio_df = commute_ratio_df.merge(monthly_evening, on='month', how='left')

commute_ratio_df[['morning_total', 'evening_total']] = commute_ratio_df[['morning_total', 'evening_total']].fillna(0)

commute_ratio_df['morning_ratio'] = commute_ratio_df['morning_total'] / commute_ratio_df['month_total']
commute_ratio_df['evening_ratio'] = commute_ratio_df['evening_total'] / commute_ratio_df['month_total']
commute_ratio_df['commute_ratio_total'] = (
    commute_ratio_df['morning_total'] + commute_ratio_df['evening_total']
) / commute_ratio_df['month_total']

print(commute_ratio_df)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(commute_ratio_df['month'], commute_ratio_df['morning_ratio'], marker='o', label='출근 비중')
plt.plot(commute_ratio_df['month'], commute_ratio_df['evening_ratio'], marker='o', label='퇴근 비중')
plt.plot(commute_ratio_df['month'], commute_ratio_df['commute_ratio_total'], marker='o', label='출퇴근 합계 비중')

plt.title('2024년 월별 출퇴근 시간대 비중')
plt.xlabel('월')
plt.ylabel('비중')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(commute_ratio_df['month'], commute_ratio_df['morning_ratio'], marker='o', label='출근 비중')
plt.plot(commute_ratio_df['month'], commute_ratio_df['evening_ratio'], marker='o', label='퇴근 비중')
plt.plot(commute_ratio_df['month'], commute_ratio_df['commute_ratio_total'], marker='o', label='출퇴근 합계 비중')

plt.title('2024년 월별 출퇴근 시간대 비중')
plt.xlabel('월')
plt.ylabel('비중')
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()